# 里程碑 1：双均线交叉端到端跑通

验证链路：`akshare 取数 → 规整落盘 → 算指标 → 生成信号 → 回测 → 统计`。

标的：`000001`（平安银行），日线，后复权。

**为什么用后复权**：前复权的历史价格会在每次分红后被追溯性重写，同一段历史今天跑和
下个月跑结果不同，回测不可复现。后复权数值稳定且不含未来函数。

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
import vectorbt as vbt

import ashare
from ashare.config import DEFAULT_ADJUST, DEFAULT_FEES
from ashare.data import get_prices, trade_calendar
from ashare.strategy import MAParams, drop_limit_signals, limit_pct, ma_cross_signals
from ashare.backtest import run_signals

# 下面两行都只为控制 notebook 体积，不影响回测结果。
#
# vectorbt 的 .plot() 默认返回 ipywidgets 控件（use_widgets=True），控件状态会连同
# 整份行情数据序列化进 notebook 的 metadata.widgets，实测 10.3MB —— 且没有任何
# cell 输出引用它，纯属死重。关掉后 .plot() 返回普通 plotly Figure，缩放/悬停照常，
# 只是少了控件面板。
vbt.settings["plotting"]["use_widgets"] = False
# plotly.js 交给 CDN，不内联进每张图（默认渲染器每张图要塞 ~3.5MB 的库）。
pio.renderers.default = "notebook_connected"

print("vectorbt", vbt.__version__)
print("pandas  ", pd.__version__)
print("numpy   ", np.__version__)
print("ashare  ", ashare.__version__)

vectorbt 1.1.0
pandas   3.0.6
numpy    2.4.6
ashare   0.1.0


## 1. 取数（首次走网络，之后走 parquet 缓存）

In [2]:
SYMBOL = "000001"
START, END = "20180101", "20260919"

df = get_prices(SYMBOL, START, END, adjust=DEFAULT_ADJUST)
print(f"{len(df)} 行  {df.index.min().date()} ~ {df.index.max().date()}")
df.tail()

2116 行  2018-01-02 ~ 2026-09-18


,Open,High,Low,Close,Volume,Amount,PctChg,Turnover
Date,,,,,,,,
2026-09-14,1768.02,1793.64,1766.51,1786.10,74124802.0,8.760710e+08,0.936977,0.381975
2026-09-15,1781.58,1790.63,1774.05,1781.58,75525146.0,8.933829e+08,-0.253065,0.389191
2026-09-16,1778.57,1784.60,1743.90,1763.49,94962632.0,1.106653e+09,-1.015391,0.489355
2026-09-17,1760.48,1769.52,1743.90,1749.93,69192535.0,8.061531e+08,-0.768930,0.356558
2026-09-18,1746.91,1781.58,1742.39,1763.49,85303779.0,9.999181e+08,0.774888,0.439581


In [3]:
# 数据质量断言：索引单调、OHLC 无缺失
assert isinstance(df.index, pd.DatetimeIndex)
assert df.index.is_monotonic_increasing
assert not df.index.has_duplicates
assert df[["Open", "High", "Low", "Close"]].notna().all().all()
_cal = trade_calendar()
print("缓存区间内的交易日数:", _cal.to_series().between(df.index.min(), df.index.max()).sum())
print("实际行数            :", len(df))
print("差值即停牌天数      :", _cal.to_series().between(df.index.min(), df.index.max()).sum() - len(df))

缓存区间内的交易日数: 2116
实际行数            : 2116
差值即停牌天数      : 0


In [4]:
# 缓存命中验证：第二次调用应显著快于首次
import time

t0 = time.perf_counter()
df2 = get_prices(SYMBOL, START, END, adjust=DEFAULT_ADJUST)
elapsed = time.perf_counter() - t0

pd.testing.assert_frame_equal(df, df2)
print(f"缓存命中，耗时 {elapsed * 1000:.1f} ms")

缓存命中，耗时 3.3 ms


## 2. 价格走势

In [5]:
close = df["Close"]
close.vbt.plot(trace_kwargs=dict(name=f"{SYMBOL} 后复权收盘"), width=1000, height=400).show()

## 3. 信号

双均线交叉，并剔除落在涨跌停价上的入场信号 —— 那些价位实际挂不出单，不回剔会让
结果偏乐观。注意剔除的只是「信号当天已封板」，次日跳空封板仍然无法建模。

In [6]:
params = MAParams(fast=10, slow=50)
entries, exits = ma_cross_signals(close, params)

entries_nofilter = entries.copy()
entries = drop_limit_signals(entries, df["PctChg"], limit_pct(SYMBOL))

print(f"参数: fast={params.fast} slow={params.slow}")
print(f"原始入场信号 {entries_nofilter.sum()} 个，剔除涨跌停后 {entries.sum()} 个")
print(f"出场信号 {exits.sum()} 个")

参数: fast=10 slow=50
原始入场信号 28 个，剔除涨跌停后 27 个
出场信号 27 个


## 4. 回测

In [7]:
# SHIFT=1：信号在次一根 K 线收盘成交，满足 T+1。
# 若写 0，vectorbt 会拿信号当根的收盘价成交 —— 而当日收盘价要等收盘才知道，
# 等于用未来信息下单。run_signals 的默认值已是 1，这里显式写出以便看见。
SHIFT = 1

pf = run_signals(close, entries, exits, shift=SHIFT)
print(f"初始资金 {pf.init_cash:,.0f}   费率 {DEFAULT_FEES}（对称，近似双边成本）")
print(f"成交时点 信号后第 {SHIFT} 根 K 线")
pf.stats()

初始资金 100,000   费率 0.0008（对称，近似双边成本）
成交时点 信号后第 1 根 K 线


Start                             2018-01-02 00:00:00
End                               2026-09-18 00:00:00
Period                             2116 days 00:00:00
Start Value                                  100000.0
End Value                               134837.925577
Total Return [%]                            34.837926
Benchmark Return [%]                        11.603402
Max Gross Exposure [%]                          100.0
Total Fees Paid                           5997.998968
Max Drawdown [%]                            39.240218
Max Drawdown Duration              1363 days 00:00:00
Total Trades                                       27
Total Closed Trades                                26
Total Open Trades                                   1
Open Trade PnL                             8196.29797
Win Rate [%]                                30.769231
Best Trade [%]                               51.28008
Worst Trade [%]                            -10.581403
Avg Winning Trade [%]       

In [8]:
assert pf.stats()["Total Trades"] > 0, "没有产生任何交易，检查信号或数据"
assert np.isfinite(pf.total_return()), "总收益不是有限值"

trades = pf.trades.records_readable
assert (trades["Exit Timestamp"] >= trades["Entry Timestamp"]).all(), "存在平仓早于开仓的记录"
print(f"{len(trades)} 笔交易，断言全部通过")
trades.head(10)

27 笔交易，断言全部通过


,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
0,0,0,81.773669,2018-08-27,1221.91,79.936051,2018-11-28,1204.33,78.785986,-1596.303139,-0.015976,Long,Closed,0
1,1,0,81.642936,2019-01-22,1204.33,78.660029,2019-05-15,1513.62,98.861104,25073.822469,0.255010,Long,Closed,1
2,2,0,77.918706,2019-06-26,1583.43,98.703053,2019-11-27,1832.14,114.206382,19166.251877,0.155345,Long,Closed,2
3,3,0,73.832778,2019-12-25,1930.44,114.023798,2020-02-04,1729.10,102.131405,-15081.646667,-0.105814,Long,Closed,3
4,4,0,76.873994,2020-05-11,1658.04,101.968125,2020-06-22,1522.51,93.633139,-10614.333622,-0.083276,Long,Closed,4
5,5,0,68.415871,2020-08-13,1708.00,93.483446,2021-03-12,2587.30,141.609906,59922.981917,0.512801,Long,Closed,5
6,6,0,62.836223,2021-04-28,2812.54,141.383511,2021-06-25,2835.88,142.556789,1182.657133,0.006692,Long,Closed,6
7,7,0,81.507777,2021-09-22,2182.75,142.328881,2021-09-24,2106.27,137.341909,-6513.385604,-0.036610,Long,Closed,7
8,8,0,72.146261,2021-10-19,2375.77,137.122338,2021-11-09,2128.12,122.828720,-18126.972546,-0.105756,Long,Closed,8
9,9,0,79.917856,2022-04-13,1918.10,122.632352,2022-05-10,1771.21,113.241044,-11975.007254,-0.078120,Long,Closed,9


In [9]:
pf.plot().show()

## 5. 手工核对一笔交易

抽查第一笔，确认成交价确实取自信号次日、手续费等于成交额乘以费率。

In [10]:
t = trades.iloc[0]
print(t[["Entry Timestamp", "Avg Entry Price", "Exit Timestamp",
         "Avg Exit Price", "PnL", "Entry Fees", "Exit Fees"]].to_string())

# 成交时点核对：入场日应恰好是首个信号的次一个交易日
sig_date = entries[entries].index[0]
next_bar = close.index[close.index.get_loc(sig_date) + 1]
print(f"\n首个入场信号日 {sig_date.date()}")
print(f"次一个交易日   {next_bar.date()}")
print(f"交易实际入场日 {t['Entry Timestamp'].date()}")
assert t["Entry Timestamp"] == next_bar, "成交时点没有顺延一根 K 线，存在未来函数"
assert abs(t["Avg Entry Price"] - close.loc[next_bar]) < 1e-6, "成交价不等于该日收盘价"

# 手续费核对：成交额 × 费率
notional = t["Size"] * t["Avg Entry Price"]
expected_fee = notional * DEFAULT_FEES
print(f"\n买入成交额 {notional:,.2f} × 费率 {DEFAULT_FEES} = {expected_fee:,.2f}")
print(f"记录中的 Entry Fees                    = {t['Entry Fees']:,.2f}")
assert abs(expected_fee - t["Entry Fees"]) < 1.0, "手续费与 成交额×费率 不符"
print("成交时点与手续费核对均通过")

Entry Timestamp    2018-08-27 00:00:00
Avg Entry Price                1221.91
Exit Timestamp     2018-11-28 00:00:00
Avg Exit Price                 1204.33
PnL                       -1596.303139
Entry Fees                   79.936051
Exit Fees                    78.785986

首个入场信号日 2018-08-24
次一个交易日   2018-08-27
交易实际入场日 2018-08-27

买入成交额 99,920.06 × 费率 0.0008 = 79.94
记录中的 Entry Fees                    = 79.94
成交时点与手续费核对均通过


## 6. 已知偏差（解读结果时必须考虑）

1. **涨跌停**：开源版 vectorbt 无限价单，只能按信号价成交。信号日封板的单已剔除，但
   次日跳空封板仍无法成交 —— 结果仍偏乐观。
2. **费率对称**：真实成本是佣金（约 0.0003，有 5 元起收）+ 印花税 0.0005（仅卖出）
   + 过户费 0.00001。`fees=0.0008` 对称征收，大致匹配双边总额但高估了买入腿。
   小资金、高换手策略受此影响更大。
3. **无滑点**：成交价直接取收盘价，未建模冲击成本和买卖价差。
4. **成交时点**：`run_signals` 默认 `shift=1`，信号在次一根 K 线收盘成交，满足 T+1。
   **`shift=0` 是未来函数** —— 当日收盘价要等收盘才知道，拿它成交等于用未来信息
   下单。本 notebook 早先版本就是 `shift=0`，总收益 25.28%；改为 `shift=1` 后见上文
   回测输出，两者相差近 10 个百分点，量级足以让结论失真。别把 `shift` 调回 0。
5. **停牌**：停牌日在缓存中不存在（非 NaN 填充），vectorbt 不会在停牌 bar 上成交，
   但持仓市值会跨停牌期连续计算。
6. **单只单参数**：本 notebook 只验证链路，未做样本内外分割。换参数直到好看就是
   过拟合，参数寻优必须配合 `vbt.rolling_split()`。
7. **数据来源不唯一**：`fetch_hist` 在 eastmoney → sina → tencent 之间按序回退，
   实测东财会限流。三家口径已在本层统一（涨跌幅自算、换手率统一为百分数），但
   成交额/换手率的原始定义仍有细微差别。缓存元信息里的 `source` 字段记录了每个
   区间实际来自哪家，跨源拼接的长区间要注意这一点 —— 本次运行的 source 见上文
   取数单元格输出。